<a href="https://colab.research.google.com/github/Fareed-Ahmed-dev/skills-getting-started-with-github-copilot/blob/main/trainGpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

In [5]:
import math

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0
        self.head_size = self.n_embd // self.n_head
        self.num_attention_heads = self.n_head
        self.hidden_size = self.n_embd
        self.intermediate_size = 4 * self.n_embd
        self.max_position_embeddings = self.block_size
        self.initializer_range = 0.02
        self.rms_norm_eps = 1e-6

        self.sliding_window = None
        self.rope_theta = 10000.0
        self.rope_scaling = None

        self.bos_token_id = None
        self.eos_token_id = None

        self.tie_word_embeddings = False

This `GPTConfig` dataclass provides a structured way to define the model's hyper-parameters, including the requested `block_size`, `vocab_size`, `n_layer`, `n_head`, and `n_embd`. It also includes other common configuration parameters for a GPT-like model and a `__post_init__` method to derive related values.

In [36]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a single batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.bias = config.bias # This 'bias' refers to the config.bias boolean
        # flash attention make GPU go brrrrrrr
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: Using slow attention. Flash Attention requires PyTorch >= 2.0")
        # causal mask to ensure that attention is only applied to the left in the input sequence
        # Renamed 'bias' buffer to 'attn_mask' to avoid conflict with self.bias = config.bias
        self.register_buffer("attn_mask", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention cuda kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            # Use the renamed 'attn_mask' buffer
            att = att.masked_fill(self.attn_mask[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, eps=1e-5)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, eps=1e-5)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, eps=1e-5),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=config.bias)

        # initialize all weights
        self.apply(self._init_weights)
        # apply weight tying (GPT-2 implementation) - when using hf for pre-training, set this to true
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying
        # report number of parameters
        print(f"number of parameters: {sum(p.numel() for p in self.parameters())/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

In [7]:
from transformers import AutoConfig, AutoModelForCausalLM

First, we'll load the configuration for `gpt2` from Hugging Face. Then, we will create an instance of our `GPTConfig` class, mapping the relevant parameters from the Hugging Face configuration.

In [24]:
# Load a pre-trained GPT-2 configuration from Hugging Face
hf_config = AutoConfig.from_pretrained("gpt2")

# Create an instance of our custom GPTConfig using parameters from the Hugging Face config
gpt_config = GPTConfig(
    block_size=hf_config.max_position_embeddings,
    vocab_size=hf_config.vocab_size,
    n_layer=hf_config.n_layer,
    n_head=hf_config.n_head,
    n_embd=hf_config.n_embd,
    dropout=hf_config.attn_pdrop,
    bias=True # GPT-2 model uses bias in its linear layers
)

print(gpt_config)

GPTConfig(block_size=1024, vocab_size=50257, n_layer=12, n_head=12, n_embd=768, dropout=0.1, bias=True)


Now that we have our `gpt_config` object, we can initialize our `GPT` model with these parameters.

In [9]:
# Initialize our custom GPT model with the loaded configuration
model = GPT(gpt_config)

# Print the model to see its architecture
print(model)

number of parameters: 162.95M
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=False)
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear

Now, we will load the pre-trained `gpt2` model from Hugging Face and copy its weights to our custom `GPT` model. This involves mapping the parameter names from the Hugging Face model to our model's structure.

In [34]:
# Load the pre-trained GPT-2 model (with weights) from Hugging Face
hf_model = AutoModelForCausalLM.from_pretrained("gpt2")

# Get the state_dict from the Hugging Face model
state_dict = hf_model.state_dict()

# Create a new state_dict for our custom model with transposed weights where necessary
transposed_state_dict = {}
for k, v in state_dict.items():
    # Identify linear layer weights that need transposing.
    # PyTorch's nn.Linear expects (out_features, in_features) for weights.
    # Hugging Face GPT-2's attention and MLP weights are (in_features, out_features) and need transposition.
    # However, Hugging Face GPT-2's lm_head.weight is typically (vocab_size, n_embd), matching PyTorch's nn.Linear.
    if any(s in k for s in ['attn.c_attn.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']):
        # Transpose the weight tensor and make it contiguous to ensure proper shape for loading
        transposed_state_dict[k] = v.transpose(0, 1).contiguous()
    else:
        # For biases, embeddings, layer norm weights, AND lm_head.weight, direct copy is fine (no transposition)
        transposed_state_dict[k] = v

# Load the transposed state dictionary into our custom model
# With 'strict=False', it will ignore keys that do not match between the two models
# (e.g., if our model has fewer layers or different buffers) which is fine here.
model.load_state_dict(transposed_state_dict, strict=False)

print("Model weights loaded successfully from Hugging Face GPT-2 after robustly fixing transposing weights!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model weights loaded successfully from Hugging Face GPT-2 after robustly fixing transposing weights!


In [35]:
num_return_sequences = 5
max_length = 30 # Increased max_length for longer generation

# The model 'model' has already been initialized and its weights loaded
# from a pre-trained GPT-2 in previous cells. We do not need to call
# 'GPT.from_pretrained' as it's not a method of our custom GPT class.
# We just need to ensure the existing model is in evaluation mode and on CUDA.

# Determine the device to use (GPU if available, otherwise CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model.eval()
model.to(device)

#prefix tokens
from transformers import AutoTokenizer # Import AutoTokenizer

# Use AutoTokenizer to load the GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokens = tokenizer.encode("Hello, I'm a language model,") # Use tokenizer.encode
tokens= torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) # (5,8)
x = tokens.to(device)
import torch
import torch.nn.functional as F

torch.manual_seed(42)
torch.cuda.manual_seed(42)

temperature = 1.0 # Increased temperature for more diversity
top_p = 0.9       # Add top-p sampling parameter

while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits, _ = model(x[:, -gpt_config.block_size:]) # crop input to the block_size
        # take the logits at the last position
        logits = logits[:, -1, :]
        # apply temperature to logits
        logits = logits / temperature
        # get probabilities
        probs = F.softmax(logits, dim=-1)

        # Corrected Top-P Sampling Logic
        # Sort probabilities in descending order
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)

        # Compute cumulative probabilities
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # Create a mask for tokens to remove based on top_p
        # `indices_to_remove` will be True for tokens whose cumulative probability is > top_p
        indices_to_remove = cumulative_probs > top_p

        # Shift the `indices_to_remove` mask to the right by one position,
        # and ensure the first element (highest probability) is always False.
        # This ensures that tokens whose cumulative probability sum up to `top_p` are kept,
        # including the token that causes the cumulative sum to exceed `top_p`.
        if indices_to_remove.shape[1] > 1:
            indices_to_remove[..., 1:] = indices_to_remove[..., :-1].clone()
        indices_to_remove[..., 0] = False # Ensure at least the top-1 token is kept.

        # Apply the mask to the sorted probabilities (set to 0.0 for tokens to be removed)
        sorted_probs.masked_fill_(indices_to_remove, 0.0)

        # Scatter the filtered sorted_probs back to the original vocabulary dimension
        # `probs` now contains probabilities with top-p filtering applied.
        probs = torch.zeros_like(probs).scatter_(-1, sorted_indices, sorted_probs)

        # Renormalize the probabilities to sum to 1
        # Add a small epsilon to avoid division by zero if all probs somehow became zero.
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp(min=1e-10)

        # Sample from the (possibly pruned and rescaled) probabilities
        ix = torch.multinomial(probs, num_samples=1)

        # append to the sequence
        x = torch.cat((x, ix), dim=1)

# decode the generated tokens
for i in range(num_return_sequences):
    tokens_list = x[i, :].tolist()
    decoded_text = tokenizer.decode(tokens_list) # Use tokenizer.decode
    print(f"Generated {i+1}: {decoded_text}")

Using device: cpu
Generated 1: Hello, I'm a language model, folders Request failed once bacon recenteeelve cost car
 were think Wymine Credit disposed Cab opensforestation's
Generated 2: Hello, I'm a language model, Daddyuh, again outbreak Once Americandr c sm shaken In wond played fight confidence Chance personoked senses 49.
Generated 3: Hello, I'm a language model,IS has- would getkey luck made. simply concern. unit gn stocked TABLE/ student resist clut designsobject
Generated 4: Hello, I'm a language model, a more weap weARE AC SalFail men his's instanceFs strat Ald simpleNote tried mess Web arguments before
Generated 5: Hello, I'm a language model, guessedchardnext happens)." – several librarieskeershanded ( satire proposals full called sys 46 ne * vast recess


### Text Generation using Hugging Face's `hf_model`

To diagnose the source of the repetitive output, let's try generating text directly using the `hf_model` (the pre-trained GPT-2 model from Hugging Face) that we loaded earlier. This will help us determine if the issue is with the custom `GPT` model's implementation/weight loading, or if it lies within the sampling logic itself.

In [23]:
# Ensure the Hugging Face model is in evaluation mode and on the correct device
hf_model.eval()
hf_model.to(device)

# Re-initialize the input tokens using the same prefix
tokens = tokenizer.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)
x_hf = tokens.to(device) # Use a new variable for HF model input

print(f"Generating text with Hugging Face's GPT-2 model (max_length={max_length}, temperature={temperature}, top_p={top_p})...")

# Generation loop using the Hugging Face model
while x_hf.size(1) < max_length:
    with torch.no_grad():
        # Forward the Hugging Face model. Access logits from the output object.
        # Crop input to the model's max position embeddings
        hf_outputs = hf_model(x_hf[:, -hf_config.max_position_embeddings:])
        logits = hf_outputs.logits[:, -1, :]

        # Apply temperature to logits
        logits = logits / temperature
        # Get probabilities
        probs = F.softmax(logits, dim=-1)

        # Corrected Top-P Sampling Logic
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        indices_to_remove = cumulative_probs > top_p

        if indices_to_remove.shape[1] > 1:
            indices_to_remove[..., 1:] = indices_to_remove[..., :-1].clone()
        indices_to_remove[..., 0] = False # Ensure at least the top-1 token is kept.

        sorted_probs.masked_fill_(indices_to_remove, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, sorted_indices, sorted_probs)
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp(min=1e-10)

        # Sample from the (possibly pruned and rescaled) probabilities
        ix = torch.multinomial(probs, num_samples=1)

        # Append to the sequence
        x_hf = torch.cat((x_hf, ix), dim=1)

# Decode and print the generated tokens from the Hugging Face model
print("\n--- Generated Text (from Hugging Face model) ---")
for i in range(num_return_sequences):
    tokens_list = x_hf[i, :].tolist()
    decoded_text_hf = tokenizer.decode(tokens_list)
    print(f"Generated {i+1} (HF): {decoded_text_hf}")

Generating text with Hugging Face's GPT-2 model (max_length=100, temperature=1.0, top_p=0.9)...

--- Generated Text (from Hugging Face model) ---
Generated 1 (HF): Hello, I'm a language model, and what they've done is… The Python language is just what you expect it to be."

Kiwi he responded. "Because you need that as well, I mean you won't find any fault with people making nice and broad niceties, and I wouldn't care if you went out there and were smart and snarky about those things. You know what I mean? The Java language. So, as you're saying, you're
Generated 2 (HF): Hello, I'm a language model, so that's something I look at often. [Laughs] I'm really excited to meet a language model, but I don't know much about it yet. Theres this idea that there's something in it that makes this all have a different genesis? I'm starting to figure out some ways to articulate it. I want to explain some of the things that it does, but hopefully some of it comes from that. I'll talk more about it ne